# Feature Engineering and Selection Workflow
### Internship Track: Week 2 – Task 3
**Author:** Sayantani Adhikary  
**Objective:** Transform raw borrower data through domain-specific ratio engineering, categorical one-hot encoding, numerical standardization, and ANOVA-based statistical feature selection.


## 1. Import Libraries & Load Raw Data
We import Pandas, NumPy, Scikit-Learn modules for preprocessing and feature selection, and Seaborn for plotting.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

sns.set_theme(style="whitegrid")
df = pd.read_csv('../data/raw_dataset.csv')
print(f"Raw Dataset Shape: {df.shape}")
df.head()


## 2. Feature Creation: Domain-Specific Financial Ratios
We engineer two leverage ratios and one total exposure feature:
1. `Debt_to_Income`: Existing Debt divided by Annual Income
2. `Loan_to_Income`: Requested Loan Amount divided by Annual Income
3. `Total_Liability`: Total combined debt exposure


In [1]:
# Create interaction features
df['Debt_to_Income'] = df['ExistingDebt'] / df['AnnualIncome']
df['Loan_to_Income'] = df['LoanAmount'] / df['AnnualIncome']
df['Total_Liability'] = df['LoanAmount'] + df['ExistingDebt']

df[['AnnualIncome', 'ExistingDebt', 'Debt_to_Income', 'Loan_to_Income']].head()


## 3. Categorical Encoding (One-Hot Encoding)
Encode `EmploymentType` using one-hot encoding with `drop_first=True` to prevent multicollinearity.


In [1]:
X_raw = df.drop(columns=['CustomerID', 'LoanApproved'])
y = df['LoanApproved']

X_encoded = pd.get_dummies(X_raw, columns=['EmploymentType'], drop_first=True)
print(f"Encoded feature columns: {list(X_encoded.columns)}")


## 4. Numerical Standardization (StandardScaler)
Standardize all numeric features to zero mean and unit variance so high-magnitude values do not dominate the model.


In [1]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_encoded), columns=X_encoded.columns)
X_scaled.describe().round(2)


## 5. Statistical Feature Selection (SelectKBest with ANOVA F-Test)
Rank features by their linear discriminative power between approved and rejected loans.


In [1]:
selector = SelectKBest(score_func=f_classif, k=5)
X_selected = selector.fit_transform(X_scaled, y)

scores_df = pd.DataFrame({
    'Feature': X_scaled.columns,
    'F_Score': selector.scores_,
    'p_value': selector.pvalues_
}).sort_values(by='F_Score', ascending=False)

# Plot Feature Importance
plt.figure(figsize=(9, 4.5))
sns.barplot(x='F_Score', y='Feature', data=scores_df, palette='mako')
plt.title('ANOVA F-Statistic Feature Importance Ranking', fontsize=12, fontweight='bold')
plt.xlabel('F-Score (Higher = More Predictive)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

selected_cols = X_scaled.columns[selector.get_support()]
print(f"Top 5 Selected Features: {list(selected_cols)}")


## 6. Model Performance Comparison
Compare baseline Logistic Regression accuracy before vs. after feature engineering and selection.


In [1]:
# Baseline model on raw unscaled features
X_base = X_encoded[['Age', 'AnnualIncome', 'LoanAmount', 'ExistingDebt', 'CreditScore']]
m_base = LogisticRegression(max_iter=200).fit(X_base, y)
acc_base = accuracy_score(y, m_base.predict(X_base))

# Engineered & selected features model
m_eng = LogisticRegression(max_iter=200).fit(X_selected, y)
acc_eng = accuracy_score(y, m_eng.predict(X_selected))

print(f"Baseline Accuracy (Raw Features): {acc_base * 100:.1f}%")
print(f"Optimized Accuracy (Engineered + Selected Features): {acc_eng * 100:.1f}%")


## 7. Export Engineered Dataset
Save the final processed dataset to `data/engineered_dataset.csv`.


In [1]:
final_df = pd.DataFrame(X_selected, columns=selected_cols)
final_df['LoanApproved'] = y.values
final_df.to_csv('../data/engineered_dataset.csv', index=False)
print("Engineered dataset successfully saved to data/engineered_dataset.csv!")
